# 02: Burn Area Vectorization
 
This notebook converts the binary raster burn mask generated in Phase 1 into discrete vector polygons. Vectorization is a prerequisite for infrastructure damage assessment, as it allows us to perform spatial intersections between the burned zones and infrastructure datasets (roads and buildings). 

We utilize `rasterio.features` to extract the geometries and `geopandas` to clean, filter, and export the final GeoDataFrame.

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
import matplotlib.pyplot as plt

print("Vectorization libraries loaded successfully.")

## 2. Configuration & File Paths
We load the thresholded mask from the previous notebook. The output will be saved as a GeoJSON, which is standard for web mapping and easily readable by both Python and QGIS.

In [ ]:
# --- File Paths ---
DATA_DIR = "data"

# Input from Notebook 01
INPUT_MASK = os.path.join(DATA_DIR, "output_burn_mask_aoi.tif")

# Output Vector File
OUTPUT_VECTOR = os.path.join(DATA_DIR, "burned_area_polygons.geojson")

print("File paths configured.")

## 3. Raster to Vector Conversion
We read the raster array and its affine transform. The `rasterio.features.shapes` function traces the boundaries of contiguous pixels possessing the same value. By applying a mask (`mask_array == 1`), we ensure only the burned pixels are converted into polygons.

In [ ]:
# Extract shapes from the raster
with rasterio.open(INPUT_MASK) as src:
    mask_array = src.read(1)
    transform = src.transform
    crs = src.crs
    
    # Generate polygon geometries (only where pixel value is 1)
    # shapes() returns a generator of (polygon, value) tuples
    results = (
        {'properties': {'raster_val': v}, 'geometry': s}
        for i, (s, v) 
        in enumerate(shapes(mask_array, mask=(mask_array == 1), transform=transform))
    )

print("Pixel tracing complete.")

## 4. GeoDataFrame Creation & Cleaning
We convert the generated geometries into a `geopandas.GeoDataFrame`. 

*Note: Raw raster vectorization often produces "salt and pepper" noise (tiny 1-pixel polygons). We calculate the area of each polygon and filter out anything smaller than 2000 sq meters (5 Sentinel-2 pixels) to keep the data clean and computationally efficient.*

In [ ]:
# Convert to a list of Shapely geometries
geometries = list(results)

# Create GeoDataFrame
gdf = gpd.GeoDataFrame.from_features(geometries, crs=crs)

print(f"Original polygon count: {len(gdf)}")

# Clean up noise: Filter out extremely small isolated artifacts (e.g., < 2000 sqm)
# We temporarily project to a metric CRS if not already, to calculate accurate area
gdf['area_sqm'] = gdf.geometry.area
gdf_clean = gdf[gdf['area_sqm'] >= 2000].copy()

# Drop the temporary area column
gdf_clean = gdf_clean.drop(columns=['area_sqm'])

print(f"Cleaned polygon count: {len(gdf_clean)}")

## 5. Exporting Results
The cleaned GeoDataFrame is exported to the `data/` directory. This file is now ready for spatial joins with infrastructure data.

In [ ]:
# Export to GeoJSON
# If the file already exists, GeoPandas will overwrite it by default if using GeoJSON
gdf_clean.to_file(OUTPUT_VECTOR, driver="GeoJSON")

print(f"Vector polygons successfully saved to: {OUTPUT_VECTOR}")

## 6. Visualization
A quick plot of the generated geometries to ensure spatial integrity.

In [ ]:
# Plot the vectorized burn scars
fig, ax = plt.subplots(figsize=(10, 10))

gdf_clean.plot(ax=ax, facecolor='red', edgecolor='black', alpha=0.7, linewidth=0.5)

ax.set_title(f"Vectorized Burn Scars\nTotal Polygons: {len(gdf_clean)}")
ax.set_axis_off()

plt.tight_layout()
plt.show()